# `odeint` and `rk4` integration speed comparison

In [10]:
import sys
import time

import numpy as np
from scipy.integrate import odeint

sys.path.insert(0, "src")
from attractors.lorenz import _lorenz_attractor
from attractors.solver import solve_rk4

In [11]:
eq = _lorenz_attractor.equation
ic = np.array(_lorenz_attractor.initial_conditions, dtype=np.float64)
param_defaults = [p.default for p in _lorenz_attractor.params]
p = np.ascontiguousarray(param_defaults, dtype=np.float64)
t_min = _lorenz_attractor.time_defaults["t_min"]
t_max_app = _lorenz_attractor.time_defaults["t_max"] * 4

print("Warming up JIT")
_ = solve_rk4(eq, ic, t_min, t_max_app, 10000, p)
_ = odeint(eq, ic, np.linspace(t_min, t_max_app, 10000), args=(p,))
print("JIT warmed up")

Warming up JIT
JIT warmed up


## Count `odeint`'s actual number of integration steps

In [12]:
def counted_eq(y, t, *args):
    counted_eq.count += 1

    return eq(y, t, args[0])

n_test = [500, 1000, 5000, 10000, 50000, 100000]

for n in n_test:
    counted_eq.count = 0

    _ = odeint(
        counted_eq,
        ic,
        np.linspace(t_min, t_max_app, n),
        args=(p,),
    )

    print(f"n_out = {n} -> odeint internal evals = {counted_eq.count}")

n_out = 500 -> odeint internal evals = 57027
n_out = 1000 -> odeint internal evals = 58579
n_out = 5000 -> odeint internal evals = 57491
n_out = 10000 -> odeint internal evals = 57582
n_out = 50000 -> odeint internal evals = 57441
n_out = 100000 -> odeint internal evals = 57077


`odeint` seems to use $\approx57000$ evaluations regardless of the number of integration steps.

## Comparing odeint and solve_rk4 at equivalent work

`odeint` at `n_out = 100000` does $\approx$`X` internal evals  
`solve_rk4` at `n = X/4` does the same number (4 evals per step)  

In [13]:
n_warmup = 3
n_bench = 5

benchmarks = {
    "odeint    (n_out=100k, internal=?)  ": lambda: odeint(eq, ic, np.linspace(t_min, t_max_app, 100000), args=(p,)),
    "solve_rk4 (n=100k,     4*100k evals)": lambda: solve_rk4(eq, ic, t_min, t_max_app, 100000, p),
    "solve_rk4 (n=50k,      4*50k evals) ": lambda: solve_rk4(eq, ic, t_min, t_max_app, 50000, p),
    "solve_rk4 (n=20k,      4*20k evals) ": lambda: solve_rk4(eq, ic, t_min, t_max_app, 20000, p),
    "solve_rk4 (n=10k,      4*10k evals) ": lambda: solve_rk4(eq, ic, t_min, t_max_app, 10000, p),
    "solve_rk4 (n=5k,       4*5k evals)  ": lambda: solve_rk4(eq, ic, t_min, t_max_app, 5000, p),
}

# single solve, bifurcation does this for each around 500 param values
for label, func in benchmarks.items():
    for _ in range(n_warmup):
        func()

    times = []
    for _ in range(n_bench):
        t0 = time.perf_counter()
        func()
        times.append(time.perf_counter() - t0)

    print(f"{label}:  {np.mean(times):.4f}s +- {np.std(times):.4f}s")

odeint    (n_out=100k, internal=?)  :  0.0473s +- 0.0044s
solve_rk4 (n=100k,     4*100k evals):  0.0282s +- 0.0008s
solve_rk4 (n=50k,      4*50k evals) :  0.0146s +- 0.0004s
solve_rk4 (n=20k,      4*20k evals) :  0.0055s +- 0.0000s
solve_rk4 (n=10k,      4*10k evals) :  0.0028s +- 0.0000s
solve_rk4 (n=5k,       4*5k evals)  :  0.0014s +- 0.0001s


## Check if lower n RK4 solutions stay on the attractor

Need to detect z midplane crossings reliably and compare trajectories at various n

In [14]:
solve_odeint = odeint(eq, ic, np.linspace(t_min, t_max_app, 100000), args=(p,))

results = {}
for n in [100000, 50000, 20000, 10000, 5000]:
    sol = solve_rk4(eq, ic, t_min, t_max_app, n, p)
    # transient removal then find z midplane crossings (see bifurcation_worker.py)
    start = int(len(sol) * 0.7)
    data = sol[start:]
    z = data[:, 2]
    z_mid = (z.max() + z.min()) / 2
    crossings = np.where((z[:-1] < z_mid) & (z[1:] >= z_mid))[0]
    frac = (z_mid - z[crossings]) / (z[crossings + 1] - z[crossings])
    peaks = data[crossings, 0] + frac * (data[crossings + 1, 0] - data[crossings, 0])
    results[n] = peaks

    print(f"n={n:>6d}: {len(peaks):>4d} crossings,  mean x={np.mean(peaks):.4f}")

# reference from odeint
start = int(len(solve_odeint) * 0.7)
data = solve_odeint[start:]
z = data[:, 2]
z_mid = (z.max() + z.min()) / 2
crossings = np.where((z[:-1] < z_mid) & (z[1:] >= z_mid))[0]
frac = (z_mid - z[crossings]) / (z[crossings + 1] - z[crossings])
peaks_ref = data[crossings, 0] + frac * (data[crossings + 1, 0] - data[crossings, 0])

print(f"odeint: {len(peaks_ref)} crossings,  mean x={np.mean(peaks_ref):.4f}")

n=100000:   79 crossings,  mean x=0.0670
n= 50000:   81 crossings,  mean x=0.7192
n= 20000:   79 crossings,  mean x=0.3233
n= 10000:   80 crossings,  mean x=-0.1227
n=  5000:   79 crossings,  mean x=0.8893
odeint: 79 crossings,  mean x=-2.2455


## Extrapolate to full bifurcation sweep

In [15]:
# 500 vals x 2 sweep directions
total_solves = 1000

n = 10000
for _ in range(n_warmup):
    solve_rk4(eq, ic, t_min, t_max_app, n, p)

times = []

for _ in range(n_bench):
    t0 = time.perf_counter()
    solve_rk4(eq, ic, t_min, t_max_app, n, p)
    times.append(time.perf_counter() - t0)

per_solve = np.mean(times)

print(f"solve_rk4 n={n}: {per_solve*1000:.2f}ms per solve")
print(f"Estimated total: {per_solve * total_solves:.1f}s for {total_solves} solves")
print(f"odeint n=100k: approx {per_solve * total_solves * 4:.1f}s")

solve_rk4 n=10000: 3.42ms per solve
Estimated total: 3.4s for 1000 solves
odeint n=100k: approx 13.7s
